## **Introduction**
We set up a script to run automatically several experiments

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
## Libraries

from sklearn.datasets import make_classification
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.subplots as sp
import plotly.graph_objects as go
import seaborn as sns

# Model Training and evaluation
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import joblib
import torch
import sys
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
import lightgbm as lgb
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split

# Data augmentation SMOTE
from imblearn.over_sampling import SMOTE
from collections import Counter

import contextlib
import io
from tqdm.notebook import tqdm

#import utils
sys.path.append('/content/gdrive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/Simulations/utils')
from utils import (augment_data, plot_class_distribution, evaluate_model,prepare_data, set_scores, get_APS_scores_all,
get_RAPS_scores_all, standard_conformal, classwise_conformal, clustered_conformal, plot_feature_distributions,
                   plot_feature_distributions_interactive) #augment_with_sdv

# Split 'CovGap' and 'AvgSize' columns into separate mean and standard error columns
def split_mean_se(value):
    mean, se = value.strip(')').split(' (')
    return float(mean), float(se)


In [3]:
class CFG:
    def __init__(self):
        self.n_samples = 6000
        self.n_features = 6
        self.n_informative = 5
        self.n_redundant = 0
        self.n_classes = 8
        self.n_clusters_per_class = 1
        self.weights = None
        self.flip_y = 0.01
        self.random_state = 42
        self.class_sep = 0.5
        self.randomize = True
        self.seed = 0
        self.model = 'MLP Neural Net'
        self.n_sim = 1000
        self.alpha = 0.05
        self.score = 'APS'
        self.plot_name = 'default.pdf'
        self.scenario = 's1'
        self.match_majority_threshold = 1000 #numerosity of classes

In [4]:
def set_scores(suffix):
    global cal_scores, cal_aug_scores, test_scores

    def safe_copy(obj):
        return obj.clone() if hasattr(obj, 'clone') else obj.copy()

    cal_scores = safe_copy(globals()[f'cal_scores_{suffix}'])
    cal_aug_scores = safe_copy(globals()[f'cal_aug_scores_{suffix}'])
    test_scores = safe_copy(globals()[f'test_scores_{suffix}'])

def get_model_by_name(name, random_state):
    all_models = {
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=random_state),
        "LightGBM": lgb.LGBMClassifier(random_state=random_state),
        "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=random_state),
        "MLP Neural Net": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=random_state),
    }
    if name not in all_models:
        raise ValueError(f"Model '{name}' not recognized. Choose from: {list(all_models.keys())}")
    return all_models[name]

def train_and_evaluate(train, test, features, model_name, random_state):
    model = get_model_by_name(model_name, random_state)
    model.fit(train[features], train['labels'])

    y_pred = model.predict(test[features])
    macro_f1 = f1_score(test['labels'], y_pred, average='macro')

    print(f"{model_name} - Macro F1 Score: {macro_f1:.3f}")
    return model, macro_f1,


def simulate():

    # 1. Generate dataset
    #---------------------------------------------------------------------------------------------
    X, y = make_classification(
    n_samples=Config.n_samples,
    n_features=Config.n_features,
    n_informative=Config.n_informative,
    n_redundant=Config.n_redundant,
    n_classes=Config.n_classes,
    n_clusters_per_class=Config.n_clusters_per_class,
    weights=Config.weights,
    flip_y=Config.flip_y,
    class_sep=Config.class_sep,
    random_state=Config.random_state)

    # Covert in dataframe
    df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(Config.n_features)])
    df['labels'] = y
    features = [f'feature_{i}' for i in range(Config.n_features)]


    # 2. Train\test\calibration split
    #---------------------------------------------------------------------------------------------------------------------
    X = df.drop(columns='labels')
    y = df['labels']

    # 80% train, 20% test
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=Config.random_state)
    # 85% train, 15% cal
    X_train, X_cal, y_train, y_cal = train_test_split(X_train, y_train, stratify=y_train, test_size=0.15, random_state=Config.random_state)

    X_train.reset_index(drop=True, inplace=True)
    X_cal.reset_index(drop=True, inplace=True)
    X_test.reset_index(drop=True, inplace=True)
    y_train.reset_index(drop=True, inplace=True)
    y_cal.reset_index(drop=True, inplace=True)
    y_test.reset_index(drop=True, inplace=True)

    # New dataframe
    train = pd.DataFrame(X_train, columns=X.columns)
    test = pd.DataFrame(X_test, columns=X.columns)
    cal = pd.DataFrame(X_cal, columns=X.columns)
    train['labels'] = y_train
    test['labels'] = y_test
    cal['labels'] = y_cal

    print('train size:', train.shape)
    print('calibration size:', cal.shape)
    print('test size:', test.shape)

    # Calcola la numerosità della classe più presente nel set di calibrazione
    Config.match_majority = cal['labels'].value_counts().max() > Config.match_majority_threshold

    # Calculate the custom oversampling strategy for SMOTE.
    #Rules:
      # 1) If at least one class has more than target_per_class samples, all smaller classes
      # 2) will be upsampled to that maximum size.

      # If all classes have fewer than target_per_class samples, all will be upsampled to target_per_class.
    target_n = Config.match_majority_threshold
    class_counts = Counter(cal['labels'])

    if Config.match_majority > Config.match_majority_threshold:
      # If at least one class exceeds the threshold (1000) >>> then bring all the others up to max_count
        custom_strategy = {
            cls: Config.match_majority for cls, count in class_counts.items()
            if count < Config.match_majority
        }
    else:
        # If all classes are below the threshold (1000) >>> then bring them all to target_per_class
        custom_strategy = {
            cls: Config.match_majority_threshold for cls, count in class_counts.items()
            if count < Config.match_majority_threshold
        }

    #custom_strategy = {cls: target_n for cls, count in class_counts.items() if count < target_n}

    # 3. Data augmentation (SMOTE)
    #------------------------------------------------------------------------------------------------------------------------
    cal_aug = augment_data(X=cal[features], y=cal['labels'],
                          method='smote', #'kmeans',  # o 'smote', 'borderline', 'adasyn'
                          #match_majority=False,  # oppure False
                          match_majority=Config.match_majority,
                          custom_strategy=custom_strategy,
                          #custom_strategy={0: 1000, 1: 1000, 2: 1000, 3: 1000, 4: 1000, 5: 1000, 6: 1000, 7: 1000},  # opzionale
                          random_state=Config.random_state,
                          feature_names=features)
    cal_aug = cal_aug.sample(frac=1, random_state=42).reset_index(drop=True)

    print('calibration augmented size:', cal_aug.shape)

    # 4. Model Train and Evaluation
    #----------------------------------------------------------------------------------------------------------------------
    model, macro_f1 = train_and_evaluate(train, test, features, Config.model, Config.random_state)

    # 6. Compute probabilities
    #---------------------------------------------------------------------------------------------------------------------
    cal_probs = model.predict_proba(cal[features])
    cal_aug_probs = model.predict_proba(cal_aug[features])
    test_probs = model.predict_proba(test[features])

    # 7. Compute conformal scores
    #----------------------------------------------------------------------
    cal_labels, cal_smx = prepare_data(cal, cal_probs)
    cal_aug_labels, cal_aug_smx = prepare_data(cal_aug, cal_aug_probs)
    test_labels, test_smx = prepare_data(test, test_probs)

    # 8. Simulations
    #--------------------------------------------------------------------
    #alpha_value = Config.alpha
    numclasses = len(cal['labels'].unique())
    #n_simulations = Config.n_sim
    results_aug = []

    for i in tqdm(range(Config.n_sim), desc="Simulazioni"):
        Config.seed = i

      # 9. Compute the scores with different methods
      #-------------------------------------------------------------------------------------------------------------
        cal_scores_APS = get_APS_scores_all(cal_smx, randomize=Config.randomize, seed=Config.seed)
        cal_aug_scores_APS = get_APS_scores_all(cal_aug_smx, randomize=Config.randomize, seed=Config.seed)
        test_scores_APS = get_APS_scores_all(test_smx, randomize=Config.randomize, seed=Config.seed)

        cal_scores_RAPS = get_RAPS_scores_all(cal_smx, lmbda=.01, kreg=5, randomize=Config.randomize, seed=Config.seed)
        cal_aug_scores_RAPS = get_RAPS_scores_all(cal_aug_smx, lmbda=.01, kreg=5, randomize=Config.randomize, seed=Config.seed)
        test_scores_RAPS = get_RAPS_scores_all(test_smx, lmbda=.01, kreg=5, randomize=Config.randomize, seed=Config.seed)

        cal_scores_STD = 1 - cal_smx
        cal_aug_scores_STD = 1 - cal_aug_smx
        test_scores_STD = 1 - test_smx

        # Store the global variables required for the selected method
        if Config.score == 'APS':
          globals()['cal_scores_APS'] = cal_scores_APS
          globals()['cal_aug_scores_APS'] = cal_aug_scores_APS
          globals()['test_scores_APS'] = test_scores_APS
        elif Config.score == 'RAPS':
          globals()['cal_scores_RAPS'] = cal_scores_RAPS
          globals()['cal_aug_scores_RAPS'] = cal_aug_scores_RAPS
          globals()['test_scores_RAPS'] = test_scores_RAPS
        elif Config.score == 'STD':
          globals()['cal_scores_STD'] = cal_scores_STD
          globals()['cal_aug_scores_STD'] = cal_aug_scores_STD
          globals()['test_scores_STD'] = test_scores_STD
        else:
          raise ValueError(f"Unknown score type: {Config.score}")

        # Ora possiamo richiamare set_scores senza errori
        set_scores(Config.score)

        with contextlib.redirect_stdout(io.StringIO()):
            _, _, marginal_coverage_metrics, marginal_set_size_metrics = standard_conformal(cal_scores, cal_labels, test_scores, test_labels, alpha=Config.alpha)
            _, _, classwise_coverage_metrics, classwise_set_size_metrics = classwise_conformal(cal_scores, cal_labels, test_scores, test_labels, alpha=Config.alpha, num_classes=numclasses)
            _, _, clustered_cov_metrics, clustered_set_size_metrics = clustered_conformal(cal_scores, cal_labels, alpha=Config.alpha, val_scores_all=test_scores, val_labels=test_labels)
            _, _, coverage_metrics_aug, set_size_metrics_aug = classwise_conformal(cal_aug_scores, cal_aug_labels, test_scores, test_labels, alpha=Config.alpha, num_classes=numclasses)

        results_aug.append({
            'seed': Config.seed,
            'marginal_cov_gap': marginal_coverage_metrics['mean_class_cov_gap'],
            'marginal_set_size': marginal_set_size_metrics['mean'],
            'classwise_cov_gap': classwise_coverage_metrics['mean_class_cov_gap'],
            'classwise_set_size': classwise_set_size_metrics['mean'],
            'clust_cov_gap': clustered_cov_metrics['mean_class_cov_gap'],
            'clust_set_size': clustered_set_size_metrics['mean'],
            'class_aug_cov_gap': coverage_metrics_aug['mean_class_cov_gap'],
            'class_aug_set_size': set_size_metrics_aug['mean']
        })

    # 6. Final summary
    results_df = pd.DataFrame(results_aug)

    def format_mean_std(series):
        return f"{series.mean():.2f} ({series.std():.3f})"

    table_df = pd.DataFrame({
        "Method": ["Marginal", "Classwise", "Clustered", "Augmented"],
        "CovGap": [
            format_mean_std(results_df["marginal_cov_gap"]),
            format_mean_std(results_df["classwise_cov_gap"]),
            format_mean_std(results_df["clust_cov_gap"]),
            format_mean_std(results_df["class_aug_cov_gap"]),
        ],
        "AvgSize": [
            format_mean_std(results_df["marginal_set_size"]),
            format_mean_std(results_df["classwise_set_size"]),
            format_mean_std(results_df["clust_set_size"]),
            format_mean_std(results_df["class_aug_set_size"]),
        ]
    })

    #print(table_df) #per uso singolo modello
    return results_df, macro_f1 # per le simulazioni con più modelli

# load experiments setup
df = pd.read_csv("/content/gdrive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/Simulations/notebook/number_experiments_setup.csv")

In [5]:
configs = []  # generate a list to save any experiment

for _, row in df.iterrows():
    cfg = CFG()  # Create a new configuration for each each experiment
    cfg.n_samples = row['n_samples']
    cfg.n_features = row['n_features']
    cfg.n_informative = row['n_informative']

    cfg.class_sep = row['class_sep']
    cfg.scenario = row['scenario']
    cfg.experiment = row['experiment']
    cfg.plot_name = f"{row['experiment']}.pdf"

    # Gestione weights
    if row['weights'] == 'unbalanced':
        cfg.weights = [0.4, 0.2, 0.1, 0.08, 0.07, 0.06, 0.05, 0.04]
    elif row['weights'] == 'equal':
        cfg.weights = None
    else:
        raise ValueError(f"Unknown weights type: {row['weights']}")

    configs.append(cfg)

#Config = configs[1]
#vars(Config)

In [ ]:
vars(configs[21])

{'n_samples': 40000,
 'n_features': 200,
 'n_informative': 20,
 'n_redundant': 0,
 'n_classes': 8,
 'n_clusters_per_class': 1,
 'weights': [0.4, 0.2, 0.1, 0.08, 0.07, 0.06, 0.05, 0.04],
 'flip_y': 0.01,
 'random_state': 42,
 'class_sep': 0.8,
 'randomize': True,
 'seed': 0,
 'model': 'MLP Neural Net',
 'n_sim': 1000,
 'alpha': 0.05,
 'score': 'APS',
 'plot_name': 's2_8.pdf',
 'scenario': 's2',
 'match_majority_threshold': 1000,
 'experiment': 's2_8'}

In [6]:
if __name__ == "__main__":

    summary_rows_all = []

    #for i in range(len(configs)):    # All experiments
    #for i in range(0,16):            # part1
    #for i in range(16,21):           # part2
    #for i in range(22,len(configs)): # part3

    for i in range(21,22):
        Config = configs[i]

        print(f"\n========== Running Config {i+1} ==========")
        print(f"Scenario: {Config.scenario}, Experiment: {Config.experiment}")

        all_results = []
        model_names = ["Random Forest", "XGBoost", "MLP Neural Net"]
        model_f1_scores = {}

        for model_name in model_names:
            print(f"\nRunning simulations for: {model_name}")
            Config.model = model_name
            results_aug, macro_f1 = simulate()
            results_aug["Model"] = model_name
            all_results.append(results_aug)
            model_f1_scores[model_name] = macro_f1

        combined_df = pd.concat(all_results, ignore_index=True)

        def format_mean_std(series):
            return f"{series.mean():.2f} ({series.std():.3f})"

        summary_rows = []

        for model_name in model_names:
            df_model = combined_df[combined_df["Model"] == model_name]
            for method, cov_col, size_col in [
                ("Standard", "marginal_cov_gap", "marginal_set_size"),
                ("Classwise", "classwise_cov_gap", "classwise_set_size"),
                ("Clustered", "clust_cov_gap", "clust_set_size"),
                ("Augmented", "class_aug_cov_gap", "class_aug_set_size"),
            ]:
                summary_rows.append({
                    "Scenario": Config.scenario,
                    "Experiment": Config.experiment,
                    "Model": model_name,
                    "Method": method,
                    "CovGap": format_mean_std(df_model[cov_col]),
                    "AvgSize": format_mean_std(df_model[size_col]),
                    "F1 Score": f"{model_f1_scores[model_name]:.2f}",
                })

        summary_rows_all.extend(summary_rows)

    summary_df = pd.DataFrame(summary_rows_all)

    print("\n=== Overall Summary ===")
    print(summary_df.to_string(index=False))






========== Running Config 22 ==========
Scenario: s2, Experiment: s2_8

Running simulations for: Random Forest
train size: (27200, 201)
calibration size: (4800, 201)
test size: (8000, 201)
Distribuzione originale: Counter({0: 1907, 1: 957, 2: 482, 3: 385, 4: 341, 5: 289, 6: 243, 7: 196})
Distribuzione dopo augmentation: Counter({0: 1907, 1: 1000, 6: 1000, 2: 1000, 4: 1000, 3: 1000, 7: 1000, 5: 1000})


/content/gdrive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/Simulations/utils/utils.py:203: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['labels'] = y_aug
/content/gdrive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/Simulations/utils/utils.py:207: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['set'] = ['original'] * n_original + ['synthetic'] * (len(df) - n_original)


calibration augmented size: (8907, 202)
Random Forest - Macro F1 Score: 0.428


Simulazioni:   0%|          | 0/1000 [00:00<?, ?it/s]


Running simulations for: XGBoost
train size: (27200, 201)
calibration size: (4800, 201)
test size: (8000, 201)
Distribuzione originale: Counter({0: 1907, 1: 957, 2: 482, 3: 385, 4: 341, 5: 289, 6: 243, 7: 196})
Distribuzione dopo augmentation: Counter({0: 1907, 1: 1000, 6: 1000, 2: 1000, 4: 1000, 3: 1000, 7: 1000, 5: 1000})
calibration augmented size: (8907, 202)


...

/content/gdrive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/Simulations/utils/utils.py:203: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['labels'] = y_aug
/content/gdrive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/Simulations/utils/utils.py:207: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['set'] = ['original'] * n_original + ['synthetic'] * (len(df) - n_original)


MLP Neural Net - Macro F1 Score: 0.816


Simulazioni:   0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Overall Summary ===
Scenario Experiment          Model    Method       CovGap      AvgSize F1 Score
      s2       s2_8  Random Forest  Standard 0.09 (0.003) 3.52 (0.036)     0.43
      s2       s2_8  Random Forest Classwise 0.01 (0.002) 4.30 (0.071)     0.43
      s2       s2_8  Random Forest Clustered 0.09 (0.003) 3.53 (0.037)     0.43
      s2       s2_8  Random Forest Augmented 0.01 (0.002) 4.08 (0.032)     0.43
      s2       s2_8        XGBoost  Standard 0.02 (0.002) 1.71 (0.025)     0.82
      s2       s2_8        XGBoost Classwise 0.01 (0.002) 1.90 (0.034)     0.82
      s2       s2_8        XGBoost Clustered 0.02 (0.002) 1.70 (0.026)     0.82
      s2       s2_8        XGBoost Augmented 0.01 (0.002) 1.73 (0.018)     0.82
      s2       s2_8 MLP Neural Net  Standard 0.04 (0.001) 1.31 (0.007)     0.82
      s2       s2_8 MLP Neural Net Classwise 0.01 (0.001) 1.78 (0.019)     0.82
      s2       s2_8 MLP Neural Net Clustered 0.04 (0.002) 1.31 (0.008)     0.82
      s2       

In [7]:
summary_df = pd.DataFrame(summary_rows_all)
summary_df

,Scenario,Experiment,Model,Method,CovGap,AvgSize,F1 Score
0,s2,s2_8,Random Forest,Standard,0.09 (0.003),3.52 (0.036),0.43
1,s2,s2_8,Random Forest,Classwise,0.01 (0.002),4.30 (0.071),0.43
2,s2,s2_8,Random Forest,Clustered,0.09 (0.003),3.53 (0.037),0.43
3,s2,s2_8,Random Forest,Augmented,0.01 (0.002),4.08 (0.032),0.43
4,s2,s2_8,XGBoost,Standard,0.02 (0.002),1.71 (0.025),0.82
5,s2,s2_8,XGBoost,Classwise,0.01 (0.002),1.90 (0.034),0.82
6,s2,s2_8,XGBoost,Clustered,0.02 (0.002),1.70 (0.026),0.82
7,s2,s2_8,XGBoost,Augmented,0.01 (0.002),1.73 (0.018),0.82
8,s2,s2_8,MLP Neural Net,Standard,0.04 (0.001),1.31 (0.007),0.82
9,s2,s2_8,MLP Neural Net,Classwise,0.01 (0.001),1.78 (0.019),0.82


In [ ]:
# Apply the parsing function to extract mean and standard error
summary_df[['CovGap_mean', 'CovGap_se']] = summary_df['CovGap'].apply(lambda x: pd.Series(split_mean_se(x)))
summary_df[['AvgSize_mean', 'AvgSize_se']] = summary_df['AvgSize'].apply(lambda x: pd.Series(split_mean_se(x)))


file_path = '/content/gdrive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/Simulations/results/'
name = 'summary_01_part2.csv'

# Save in csv file
summary_df.to_csv(file_path+name, index=False)  # index=False to avoid saving the index